# Retail Targeting or Institutional Product Design? Distribution Structures in European UCITS Funds

Public replication notebook.

This notebook reproduces the main descriptive and regression evidence using the public de-identified replication dataset `ucits_public_replication_dataset_v2.xlsx`. The original LSEG data are proprietary and are not included. Management groups are anonymized as `MG001`, `MG002`, etc.; this preserves fixed effects, clustered standard errors, and within-group aggregation while removing original management-group names.

Run the notebook from the same folder as the Excel replication dataset.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import os
# Keep numerical libraries single-threaded for stable execution in lightweight replication environments.
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_FILE = Path('ucits_public_replication_dataset_v2.xlsx')
OUTPUT_DIR = Path('replication_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# ============================================================
# 1. Load public replication data
# ============================================================

df = pd.read_excel(DATA_FILE, sheet_name='data')

# Basic typing
for c in ['asset_class_clean', 'asset_type_norm', 'domicile', 'management_group_id']:
    df[c] = df[c].astype('object')

print('Dataset shape:', df.shape)
print('Main regression sample:', int(df['main_sample'].sum()))
print('Effective-income regression sample:', int(df['effective_income_sample'].sum()))
print('Retail-compatibility information available:', int(df['retail_eligible_i'].notna().sum()))


In [ ]:
# ============================================================
# 2. Helper functions
# ============================================================

from types import SimpleNamespace

def fit_lpm(formula, data, cluster_var='management_group_id'):
    """Fit an OLS/LPM model and return cluster-robust results.

    The notebook uses linear probability models because coefficients are directly
    interpretable as percentage-point differences and because the empirical design
    relies on multiple layers of fixed effects.
    """
    data = data.copy()
    for _col in ['asset_class_clean', 'asset_type_norm', 'domicile', cluster_var]:
        if _col in data.columns:
            data[_col] = data[_col].astype('object')
    base = smf.ols(formula, data=data, missing='drop').fit()
    used_index = base.model.data.row_labels
    used_data = data.loc[used_index].copy()
    robust = base.get_robustcov_results(cov_type='cluster', groups=used_data[cluster_var])

    names = base.params.index
    return SimpleNamespace(
        param_series=pd.Series(robust.params, index=names),
        bse_series=pd.Series(robust.bse, index=names),
        pvalue_series=pd.Series(robust.pvalues, index=names),
        rsquared=base.rsquared,
        nobs=base.nobs,
        formula=formula
    )

def stars(p):
    if pd.isna(p):
        return ''
    if p < 0.01:
        return '***'
    if p < 0.05:
        return '**'
    if p < 0.10:
        return '*'
    return ''

def fmt_coef(model, var):
    if var not in model.param_series.index:
        return '', ''
    coef = model.param_series[var]
    se = model.bse_series[var]
    p = model.pvalue_series[var]
    return f'{coef:.3f}{stars(p)}', f'({se:.3f})'

def regression_table(models, model_names, variables, fe_rows):
    rows = []
    for var, label in variables.items():
        coef_row = {'Variable': label}
        se_row = {'Variable': ''}
        for name, model in zip(model_names, models):
            coef, se = fmt_coef(model, var)
            coef_row[name] = coef
            se_row[name] = se
        rows.extend([coef_row, se_row])
    for label, values in fe_rows.items():
        row = {'Variable': label}
        for name, value in zip(model_names, values):
            row[name] = value
        rows.append(row)
    row = {'Variable': 'Observations'}
    for name, model in zip(model_names, models):
        row[name] = int(model.nobs)
    rows.append(row)
    row = {'Variable': 'R²'}
    for name, model in zip(model_names, models):
        row[name] = round(model.rsquared, 3)
    rows.append(row)
    return pd.DataFrame(rows)


## Table 1. Descriptive statistics and UCITS architecture

In [ ]:
# ============================================================
# 3. Table 1 - Descriptive statistics
# ============================================================

df_full = df.dropna(subset=['has_paid_dividends_i']).copy()
df_tm = df_full.dropna(subset=['retail_eligible_i']).copy()

table1_vars = {
    'has_paid_dividends_i': 'Dividend-distribution feature',
    'effective_income_i': 'Effective income structure',
    'retail_eligible_i': 'Retail compatible',
    'professional_eligible_i': 'Professional compatible',
    'eligible_counterparty_i': 'Eligible counterparty compatible',
    'log_tna': 'Log(TNA)',
    'fund_age_years': 'Fund age (years)',
    'fixed_income_i': 'Fixed Income',
    'equity_i': 'Equity',
    'mixed_assets_i': 'Mixed Assets',
    'ireland_i': 'Ireland domicile',
    'luxembourg_i': 'Luxembourg domicile',
    'uk_i': 'UK domicile'
}

def descriptive_panel(data, var_dict):
    rows = []
    for var, label in var_dict.items():
        x = pd.to_numeric(data[var], errors='coerce').dropna()
        rows.append({'Variable': label, 'N': int(x.shape[0]), 'Mean': x.mean(), 'Std. Dev.': x.std()})
    return pd.DataFrame(rows)

table1_full = descriptive_panel(df_full, table1_vars).round(3)
table1_tm = descriptive_panel(df_tm, table1_vars).round(3)

print('Panel A. Full sample')
display(table1_full)
print('Panel B. Target-market sample')
display(table1_tm)

with pd.ExcelWriter(OUTPUT_DIR / 'Table1_Descriptive_Statistics.xlsx') as writer:
    table1_full.to_excel(writer, sheet_name='Panel A - Full Sample', index=False)
    table1_tm.to_excel(writer, sheet_name='Panel B - Target Market', index=False)


## Table 2. Distribution structures across UCITS domiciles

In [ ]:
# ============================================================
# 4. Table 2 - Domicile-level distribution structures
# ============================================================

main_domiciles = ['Luxembourg', 'Ireland', 'UK', 'France', 'Austria', 'Germany']

df_table2 = df.dropna(subset=['retail_eligible_i']).copy()
df_table2 = df_table2[df_table2['domicile'].isin(main_domiciles)].copy()

table2 = (
    df_table2
    .groupby('domicile', observed=True)
    .agg(
        N=('domicile', 'size'),
        **{
            'Dividend %': ('has_paid_dividends_i', lambda x: x.mean() * 100),
            'Effective Income %': ('effective_income_i', lambda x: x.mean() * 100),
            'Retail %': ('retail_eligible_i', lambda x: x.mean() * 100),
            'Fixed Income %': ('fixed_income_i', lambda x: x.mean() * 100),
        }
    )
    .reset_index()
    .rename(columns={'domicile': 'Domicile'})
)

table2['Domicile'] = pd.Categorical(table2['Domicile'], categories=main_domiciles, ordered=True)
table2 = table2.sort_values('Domicile').round(2)

display(table2)
table2.to_excel(OUTPUT_DIR / 'Table2_Domicile_Architecture.xlsx', index=False)


## Table 3. Target-market compatibility and distribution structures

In [ ]:
# ============================================================
# 5. Table 3 - Retail compatibility and dividend distributions
# ============================================================

df_t3 = df.dropna(subset=[
    'has_paid_dividends_i', 'retail_eligible_i', 'log_tna',
    'fund_age_years', 'asset_class_clean', 'management_group_id'
]).copy()

t3_m1 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i', df_t3)
t3_m2 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean)', df_t3)
t3_m3 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t3)

table3 = regression_table(
    [t3_m1, t3_m2, t3_m3],
    ['(1) Unconditional', '(2) Controls', '(3) Mgmt. FE'],
    {
        'retail_eligible_i': 'Retail compatible',
        'log_tna': 'Log(TNA)',
        'fund_age_years': 'Fund age (years)'
    },
    {
        'Asset-class FE': ['No', 'Yes', 'Yes'],
        'Management-group FE': ['No', 'No', 'Yes']
    }
)

display(table3)
table3.to_excel(OUTPUT_DIR / 'Table3_Target_Market_Compatibility.xlsx', index=False)


## Table 4. Retail compatibility, fixed-income structures, and jurisdictional effects

In [ ]:
# ============================================================
# 6. Table 4 - Fixed-income interaction and domicile effects
# ============================================================

t4_m1 = t3_m3
t4_m2 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i + fixed_income_i + retail_eligible_i:fixed_income_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t3)
t4_m3 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t3[df_t3['fixed_income_i'] == 0])
t4_m4 = fit_lpm('has_paid_dividends_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id) + C(domicile)', df_t3)

table4 = regression_table(
    [t4_m1, t4_m2, t4_m3, t4_m4],
    ['(1) Baseline', '(2) Fixed Income\nInteraction', '(3) Excluding\nFixed Income', '(4) Domicile FE'],
    {
        'retail_eligible_i': 'Retail compatible',
        'fixed_income_i': 'Fixed Income',
        'retail_eligible_i:fixed_income_i': 'Retail × Fixed Income',
        'log_tna': 'Log(TNA)',
        'fund_age_years': 'Fund age (years)'
    },
    {
        'Asset-class FE': ['Yes', 'Yes', 'Yes', 'Yes'],
        'Management-group FE': ['Yes', 'Yes', 'Yes', 'Yes'],
        'Domicile FE': ['No', 'No', 'No', 'Yes']
    }
)

display(table4)
table4.to_excel(OUTPUT_DIR / 'Table4_Fixed_Income_Domicile_Effects.xlsx', index=False)


## Table 5. Effective-income structures, fixed-income interactions, and jurisdictional effects

In [ ]:
# ============================================================
# 7. Table 5 - Alternative payout measure based on recorded amounts
# ============================================================

df_t5 = df.dropna(subset=[
    'effective_income_i', 'retail_eligible_i', 'log_tna',
    'fund_age_years', 'asset_class_clean', 'management_group_id'
]).copy()

t5_m1 = fit_lpm('effective_income_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t5)
t5_m2 = fit_lpm('effective_income_i ~ retail_eligible_i + fixed_income_i + retail_eligible_i:fixed_income_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t5)
t5_m3 = fit_lpm('effective_income_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df_t5[df_t5['fixed_income_i'] == 0])
t5_m4 = fit_lpm('effective_income_i ~ retail_eligible_i + log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id) + C(domicile)', df_t5)

table5 = regression_table(
    [t5_m1, t5_m2, t5_m3, t5_m4],
    ['(1) Baseline', '(2) Fixed Income\nInteraction', '(3) Excluding\nFixed Income', '(4) Domicile FE'],
    {
        'retail_eligible_i': 'Retail compatible',
        'fixed_income_i': 'Fixed Income',
        'retail_eligible_i:fixed_income_i': 'Retail × Fixed Income',
        'log_tna': 'Log(TNA)',
        'fund_age_years': 'Fund age (years)'
    },
    {
        'Asset-class FE': ['Yes', 'Yes', 'Yes', 'Yes'],
        'Management-group FE': ['Yes', 'Yes', 'Yes', 'Yes'],
        'Domicile FE': ['No', 'No', 'No', 'Yes']
    }
)

display(table5)
table5.to_excel(OUTPUT_DIR / 'Table5_Effective_Income_Robustness.xlsx', index=False)


## Table 6. Institutional product engineering across anonymized management groups

In [ ]:
# ============================================================
# 8. Table 6 - Management-group heterogeneity
# ============================================================

mgmt_table = (
    df.dropna(subset=['management_group_id'])
    .groupby('management_group_id', observed=True)
    .agg(
        N=('fund_id', 'count'),
        **{
            'Dividend %': ('has_paid_dividends_i', lambda x: x.mean() * 100),
            'Effective Income %': ('effective_income_i', lambda x: x.mean() * 100),
            'Retail %': ('retail_eligible_i', lambda x: x.mean() * 100),
            'Fixed Income %': ('fixed_income_i', lambda x: x.mean() * 100),
        }
    )
    .reset_index()
    .rename(columns={'management_group_id': 'Management Group ID'})
    .sort_values('N', ascending=False)
)

panel_a = mgmt_table.head(15).round(2)
display(panel_a)

mgmt_20 = mgmt_table[mgmt_table['N'] >= 20].copy()
panel_b = mgmt_20[['Dividend %', 'Effective Income %']].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]).loc[['mean','std','10%','25%','50%','75%','90%']].T.reset_index()
panel_b = panel_b.rename(columns={'index':'Variable','mean':'Mean','std':'Std. Dev.','10%':'P10','25%':'P25','50%':'Median','75%':'P75','90%':'P90'}).round(1)
panel_b['Variable'] = panel_b['Variable'].replace({'Dividend %':'Dividend-distribution rate','Effective Income %':'Effective-income rate'})
display(panel_b)

# Incremental explanatory power of management-group FE.
div_no_mgmt = fit_lpm('has_paid_dividends_i ~ log_tna + fund_age_years + C(asset_class_clean)', df.dropna(subset=['has_paid_dividends_i','log_tna','fund_age_years','asset_class_clean','management_group_id']))
div_with_mgmt = fit_lpm('has_paid_dividends_i ~ log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df.dropna(subset=['has_paid_dividends_i','log_tna','fund_age_years','asset_class_clean','management_group_id']))
eff_no_mgmt = fit_lpm('effective_income_i ~ log_tna + fund_age_years + C(asset_class_clean)', df.dropna(subset=['effective_income_i','log_tna','fund_age_years','asset_class_clean','management_group_id']))
eff_with_mgmt = fit_lpm('effective_income_i ~ log_tna + fund_age_years + C(asset_class_clean) + C(management_group_id)', df.dropna(subset=['effective_income_i','log_tna','fund_age_years','asset_class_clean','management_group_id']))

panel_c = pd.DataFrame({
    'Dependent Variable': ['Dividend-distribution structures', 'Effective-income structures'],
    'R² without Mgmt FE': [div_no_mgmt.rsquared, eff_no_mgmt.rsquared],
    'R² with Mgmt FE': [div_with_mgmt.rsquared, eff_with_mgmt.rsquared],
})
panel_c['Increase'] = panel_c['R² with Mgmt FE'] - panel_c['R² without Mgmt FE']
panel_c = panel_c.round(3)
display(panel_c)

with pd.ExcelWriter(OUTPUT_DIR / 'Table6_Management_Group_Heterogeneity.xlsx') as writer:
    panel_a.to_excel(writer, sheet_name='Panel A', index=False)
    panel_b.to_excel(writer, sheet_name='Panel B', index=False)
    panel_c.to_excel(writer, sheet_name='Panel C', index=False)


## Appendix Table A1. Determinants of MiFID II disclosure availability

In [ ]:
# ============================================================
# 9. Appendix Table A1 - Disclosure availability
# ============================================================

df_a1 = df.dropna(subset=['disclosure_available_i','has_paid_dividends_i','log_tna','fund_age_years','asset_class_clean','domicile','management_group_id']).copy()

a1_m1 = fit_lpm('disclosure_available_i ~ has_paid_dividends_i + log_tna + fund_age_years', df_a1)
a1_m2 = fit_lpm('disclosure_available_i ~ has_paid_dividends_i + log_tna + fund_age_years + C(asset_class_clean)', df_a1)
a1_m3 = fit_lpm('disclosure_available_i ~ has_paid_dividends_i + log_tna + fund_age_years + C(asset_class_clean) + C(domicile)', df_a1)

table_a1 = regression_table(
    [a1_m1, a1_m2, a1_m3],
    ['(1) Basic', '(2) Asset-Class FE', '(3) Domicile FE'],
    {
        'has_paid_dividends_i': 'Dividend-distribution feature',
        'log_tna': 'Log(TNA)',
        'fund_age_years': 'Fund age (years)'
    },
    {
        'Asset-class FE': ['No', 'Yes', 'Yes'],
        'Domicile FE': ['No', 'No', 'Yes']
    }
)

display(table_a1)
table_a1.to_excel(OUTPUT_DIR / 'Appendix_Table_A1_Disclosure_Availability.xlsx', index=False)


## Supplemental checks

In [ ]:
# ============================================================
# 10. Supplemental checks
# ============================================================

print('Cross-tabulation of dividend-event and recorded-amount payout measures')
display(pd.crosstab(df['has_paid_dividends_i'], df['effective_income_i'], margins=True))

print('Sophistication-group distribution')
display(df['sophistication_group'].value_counts(dropna=False))

print(f'All generated tables saved in: {OUTPUT_DIR.resolve()}')
